### Agents Analytics that are trained through self play
#### ver51 stands for 5 - number of rows (4x5 board - 16 pieces), 1 - self play

*Note: all configs, parameters and hyperparameters used in this phase are available in "../implementations/ver51/configs.ipynb"*

versions guide:
- X - {0: no, 1: yes}
- 51XYYYYY, X - Is Double Net
- 51YXYYYY, X - Is Dueling Net
- 51YYXYYY, X - Is Residual Net
- 51YYYXYY, X - Is trained on canonical board
- 51YYYYXY, X - Will store k previous boards in state
- 51YYYYYX, X - Will do reward shaping

#### Training Data - 51111000

Statistics for 50k episodes of training Res-Dueling-DDQN through self play:

| Milestone | Episode | Details                       |
|---|--------:|-------------------------------|
| First beat random baseline as White |   1,000 | 31.0% win vs random           |
| First beat random baseline as Black |   1,000 | 39.5% win vs random           |
| First beat fixed checkpoint |   5,000 | 76.0% win vs fixed checkpoint |
| Peak White vs Random |  46,000 | 93.0% win                     |
| Peak Black vs Random |  29,000 | 87.0% win                     |
| Peak Black vs Fixed Checkpoint |  24,000 | 91.0% win                     |
| Lowest draw in training (self-play) | 26,000 | 34.3% draw                    |
| Draw in training (self-play) at last log | 50,000 | 45.9% draw                    |

In [5]:
from collections import Counter
import torch
from domain.configs import ROWS, COLUMNS, MAX_STEPS_PER_EPISODE
from environment.grenight_environment import GrenightEnvironment
from agent.grenight_agent import GrenightAgent

In [4]:
env = GrenightEnvironment(
    is_canonical_version=False,
    will_store_history_in_state=False,
    will_do_reward_shaping=False
)

self_play_25k = GrenightAgent(
    is_self_play=True,
    is_double_net=True,
    is_dueling_net=True,
    is_residual_net=True,
    is_bulk_update=False,
    rows=ROWS,
    columns=COLUMNS,
    num_actions=env.action_encoder.num_actions,
    num_planes=env.state_encoder.num_planes,
    device="cpu"
)

checkpoint = torch.load("../implementations/ver51/p_111000/current_implementation_ep25000.pt", map_location="cpu",weights_only=False)
self_play_25k.policy_net.load_state_dict(checkpoint["policy_state_dict"])
if self_play_25k.is_double_net:
    self_play_25k.target_net.load_state_dict(checkpoint["target_state_dict"])

self_play_50k = GrenightAgent(
    is_self_play=True,
    is_double_net=True,
    is_dueling_net=True,
    is_residual_net=True,
    is_bulk_update=False,
    rows=ROWS,
    columns=COLUMNS,
    num_actions=env.action_encoder.num_actions,
    num_planes=env.state_encoder.num_planes,
    device="cpu"
)

checkpoint = torch.load("../implementations/ver51/p_111000/current_implementation_ep50000.pt", map_location="cpu",weights_only=False)
self_play_50k.policy_net.load_state_dict(checkpoint["policy_state_dict"])
if self_play_50k.is_double_net:
    self_play_50k.target_net.load_state_dict(checkpoint["target_state_dict"])

In [10]:
outcomes = Counter()
for _ in range(1000):
    env.reset()
    env.previous_pieces_encoded_q.queue.clear()

    move_count = 0
    is_white_on_turn = True
    is_draw = False
    done = False

    while move_count < MAX_STEPS_PER_EPISODE and not done:
        is_white_on_turn = env.is_white_on_turn

        if is_white_on_turn:
            action = self_play_25k.select_action(env.get_state(), env.action_mask(), 0.05)

        else:
            action = self_play_50k.select_action(env.get_state(), env.action_mask(), 0.05)

        _, _, done, is_draw, _ = env.step(action)
        move_count += 1

    if not done:
        outcomes["truncated"] += 1
    else:
        if is_draw:
            outcomes["draw"] += 1
        else:
            outcomes["white_win" if is_white_on_turn else "black_win"] += 1

print(f"STATS OUT FROM: {1000} GAMES\n"
      f"white=51111000_25k, black=51111000_50k\n"
      f"Outcomes: {outcomes}\n")

STATS OUT FROM: 1000 GAMES
white=51111000_25k, black=51111000_50k
Outcomes: Counter({'draw': 659, 'black_win': 234, 'white_win': 107})



In [12]:
outcomes = Counter()
for _ in range(1000):
    env.reset()
    env.previous_pieces_encoded_q.queue.clear()

    move_count = 0
    is_white_on_turn = True
    is_draw = False
    done = False

    while move_count < MAX_STEPS_PER_EPISODE and not done:
        is_white_on_turn = env.is_white_on_turn

        if is_white_on_turn:
            action = self_play_50k.select_action(env.get_state(), env.action_mask(), 0.05)

        else:
            action = self_play_25k.select_action(env.get_state(), env.action_mask(), 0.05)

        _, _, done, is_draw, _ = env.step(action)
        move_count += 1

    if not done:
        outcomes["truncated"] += 1
    else:
        if is_draw:
            outcomes["draw"] += 1
        else:
            outcomes["white_win" if is_white_on_turn else "black_win"] += 1

print(f"STATS OUT FROM: {1000} GAMES\n"
      f"white=51111000_50k, black=51111000_25k\n"
      f"Outcomes: {outcomes}\n")

STATS OUT FROM: 1000 GAMES
white=51111000_50k, black=51111000_25k
Outcomes: Counter({'white_win': 592, 'draw': 308, 'black_win': 100})



#### Further Training

As shown in the logs, the lowest percentage of draws occurred around the middle of training, at the 26k episode evaluation log. The 50k episode checkpoint beat the 25k episode checkpoint in both evaluations. Since both agents were trained evenly from the start, unlike the agents trained against random, we can now fairly point to color as the indicator of who will have more wins on this small board.

I can conclude to this far self play experiment is successful, because self playing agent playing as black defeating fixed agent trained against random as white, given the color indicator favoring.

Further training will experiment with the canonical board, saving previous k boards in state and reward shaping. The baseline for further agents to meet is the current 50k checkpoint.